<a href="https://colab.research.google.com/github/santhosh-kumar1928/santhoshkumar-codeboosters-2026/blob/main/day4/day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark --quiet
print('PySpark installation complete!')

PySpark installation complete!


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month, to_date, col, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
  .appName('Day4_BigData_Sales') \
  .config('spark.sql.adaptive.enabled','true') \
  .getOrCreate()

print(f'Spark Version : {spark.version}')
print(f'SparkSession : ACTIVE')
print(f'Application : {spark.sparkContext.appName}')

Spark Version : 4.0.2
SparkSession : ACTIVE
Application : Day4_BigData_Sales


In [10]:
import pandas as pd
import numpy as np

# Create dummy data
data = {
    'id': np.arange(1000),
    'item': [f'Item_{i%20}' for i in range(1000)],
    'quantity': np.random.randint(1, 100, 1000),
    'unit_price': np.round(np.random.rand(1000) * 100, 2),
    'revenue': np.round(np.random.rand(1000) * 1000, 2),
    'timestamp': pd.to_datetime(pd.date_range(start='2023-01-01', periods=1000, freq='H'))
}
df_dummy = pd.DataFrame(data)

# Save to CSV
df_dummy.to_csv('large_sales_data.csv', index=False)

print('Dummy large_sales_data.csv created successfully.')

Dummy large_sales_data.csv created successfully.


In [11]:
df_bronze = spark.read \
  .option('header','true') \
  .option('inferSchema', 'true') \
  .csv('large_sales_data.csv')

print('===Bronze Layer - Raw Data===')
print(f'Rows:{df_bronze.count()}')
print(f'Columns:{len(df_bronze.columns)}')
print(f'Names:{df_bronze.columns}')
df_bronze.printSchema()

===Bronze Layer - Raw Data===
Rows:1000
Columns:6
Names:['id', 'item', 'quantity', 'unit_price', 'revenue', 'timestamp']
root
 |-- id: integer (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [12]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)
df_bronze = df_bronze.withColumn('unit+revenue', df_bronze['unit_price'] + df_bronze['revenue'])
print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue','unit+revenue').describe().show()

First 5 rows:
+---+------+--------+----------+-------+-------------------+
|id |item  |quantity|unit_price|revenue|timestamp          |
+---+------+--------+----------+-------+-------------------+
|0  |Item_0|72      |30.17     |974.6  |2023-01-01 00:00:00|
|1  |Item_1|78      |66.69     |179.83 |2023-01-01 01:00:00|
|2  |Item_2|32      |84.6      |86.3   |2023-01-01 02:00:00|
|3  |Item_3|88      |46.6      |843.36 |2023-01-01 03:00:00|
|4  |Item_4|12      |68.36     |160.72 |2023-01-01 04:00:00|
+---+------+--------+----------+-------+-------------------+
only showing top 5 rows

Basic statistics for numeric columns:
+-------+------------------+------------------+------------------+------------------+
|summary|          quantity|        unit_price|           revenue|      unit+revenue|
+-------+------------------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|              1000|
|   mean|            49.325| 4

In [13]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_bronze.parquet')
print('Bronze Parquet saved: sales_bronze.parquet')

import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path) /1024
  total = 0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath, f))
  return total /1024

csv_size = get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1 - (parquet_size / csv_size)) * 100

print(f'CSV Size: {csv_size:.1f} KB')
print(f'Parquet Size: {parquet_size:.1f} KB')
print(f'Reduction: {reduction:.1f}%')
print(f'\nAt 1 TB  scale: csv=1000 GB -> Parquet= {1000 * (parquet_size / 100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet
CSV Size: 45.8 KB
Parquet Size: 27.3 KB
Reduction: 40.3%

At 1 TB  scale: csv=1000 GB -> Parquet= 273 GB


In [14]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)

# Correctly add a new column 'unit_price_revenue_sum' that is the sum of 'unit_price' and 'revenue'
df_bronze = df_bronze.withColumn('unit_price_revenue_sum', F.col('unit_price') + F.col('revenue'))

print('\nFirst 5 rows with new sum column:')
df_bronze.show(5, truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+---+------+--------+----------+-------+-------------------+------------------+
|id |item  |quantity|unit_price|revenue|timestamp          |unit+revenue      |
+---+------+--------+----------+-------+-------------------+------------------+
|0  |Item_0|72      |30.17     |974.6  |2023-01-01 00:00:00|1004.77           |
|1  |Item_1|78      |66.69     |179.83 |2023-01-01 01:00:00|246.52            |
|2  |Item_2|32      |84.6      |86.3   |2023-01-01 02:00:00|170.89999999999998|
|3  |Item_3|88      |46.6      |843.36 |2023-01-01 03:00:00|889.96            |
|4  |Item_4|12      |68.36     |160.72 |2023-01-01 04:00:00|229.07999999999998|
+---+------+--------+----------+-------+-------------------+------------------+
only showing top 5 rows

First 5 rows with new sum column:
+---+------+--------+----------+-------+-------------------+------------------+----------------------+
|id |item  |quantity|unit_price|revenue|timestamp          |unit+revenue      |unit_price_revenue_sum|
+

In [15]:
df_bronze.filter(F.col("revenue")>50000).show()

+---+----+--------+----------+-------+---------+------------+----------------------+
| id|item|quantity|unit_price|revenue|timestamp|unit+revenue|unit_price_revenue_sum|
+---+----+--------+----------+-------+---------+------------+----------------------+
+---+----+--------+----------+-------+---------+------------+----------------------+



In [17]:
df_silver = df_bronze \
  .dropDuplicates() \
  .dropna(subset=['id','item','revenue']) # Changed order_id to id, product to item
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('timestamp')) # Changed order_date to timestamp, removed format string as it's already a timestamp
)

df_silver = df_silver \
.withColumn('order_year',year(col('order_date'))) \
.withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(F.col('revenue') > 40000, 'High')
    .when(F.col('revenue') > 10000, 'Medium')
    .otherwise('Low')
)

print(f'Silver layer rows: {df_silver.count()}')
print('New Columns added: order_year,order_month,revenue_category')
df_silver.select('item','revenue','order_year','order_month','revenue_category').show() # Changed product to item

Silver layer rows: 1000
New Columns added: order_year,order_month,revenue_category
+-------+-------+----------+-----------+----------------+
|   item|revenue|order_year|order_month|revenue_category|
+-------+-------+----------+-----------+----------------+
|Item_13| 985.33|      2023|          1|             Low|
|Item_19|  91.81|      2023|          1|             Low|
|Item_12| 318.49|      2023|          1|             Low|
| Item_5|  22.34|      2023|          1|             Low|
|Item_17|  90.56|      2023|          1|             Low|
| Item_4| 187.67|      2023|          1|             Low|
|Item_14| 188.79|      2023|          2|             Low|
| Item_5| 343.59|      2023|          2|             Low|
|Item_10|  115.1|      2023|          1|             Low|
| Item_6| 164.35|      2023|          1|             Low|
|Item_14| 477.48|      2023|          1|             Low|
|Item_13| 178.94|      2023|          2|             Low|
| Item_3| 236.48|      2023|          1|       

In [19]:
df_silver = df_bronze \
  .dropDuplicates() \
  .dropna(subset=['id','item','revenue']) # Corrected to use 'id' and 'item'
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('timestamp')) # Corrected to use 'timestamp' and removed format string
)

df_silver = df_silver \
.withColumn('order_year',year(col('order_date'))) \
.withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(F.col('revenue') > 40000, 'High')
    .when(F.col('revenue') > 10000, 'Medium')
    .otherwise('Low')
)

bronze_rows_count = df_bronze.count()
silver_rows_count = df_silver.count()
dropped_rows = bronze_rows_count - silver_rows_count

print(f'Initial rows in Bronze layer: {bronze_rows_count}')
print(f'Silver layer rows: {silver_rows_count}')
print(f'Number of items (rows) dropped: {dropped_rows}')
print('New Columns added: order_year,order_month,revenue_category')
df_silver.select('item','revenue','order_year','order_month','revenue_category').show(8) # Corrected to use 'item'

print('\nProduct names and number of orders:')
df_silver.groupBy('item').agg(F.countDistinct('id').alias('number_of_orders')).show() # Corrected to use 'item' and 'id'

Initial rows in Bronze layer: 1000
Silver layer rows: 1000
Number of items (rows) dropped: 0
New Columns added: order_year,order_month,revenue_category
+-------+-------+----------+-----------+----------------+
|   item|revenue|order_year|order_month|revenue_category|
+-------+-------+----------+-----------+----------------+
|Item_13| 985.33|      2023|          1|             Low|
|Item_19|  91.81|      2023|          1|             Low|
|Item_12| 318.49|      2023|          1|             Low|
| Item_5|  22.34|      2023|          1|             Low|
|Item_17|  90.56|      2023|          1|             Low|
| Item_4| 187.67|      2023|          1|             Low|
|Item_14| 188.79|      2023|          2|             Low|
| Item_5| 343.59|      2023|          2|             Low|
+-------+-------+----------+-----------+----------------+
only showing top 8 rows

Product names and number of orders:
+-------+----------------+
|   item|number_of_orders|
+-------+----------------+
|Item_12| 

In [20]:
df_silver.write \
  .mode('overwrite') \
  .parquet('sales_silver.parquet')

print('Silver Parquet saved:sales_silver.parquet')
print(f'Silver size:{get_dir_size("sales_silver.parquet"):.1f} KB')

df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Read-back rows:{df_verify.count()} (should match Silver count)')
df_verify.printSchema()

Silver Parquet saved:sales_silver.parquet
Silver size:35.5 KB
Read-back rows:1000 (should match Silver count)
root
 |-- id: integer (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- unit+revenue: double (nullable = true)
 |-- unit_price_revenue_sum: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

